# Llama 3.3 70B Instruct Binary Classification

This notebook evaluates Meta Llama 3.3 70B Instruct for binary classification of organization names.

- Class 1: Government-related funding organization
- Class 0: Non-government-related funding organization
- Prompting methods: Zero-shot and few-shot
- Evaluation method: Stratified 10-fold cross-validation
- Few-shot examples: 10 examples sampled only from each training fold

In [1]:
# Install only the additional packages required for the experiment.
%pip install -q \
    "sagemaker<3" \
    scikit-learn \
    pandas \
    numpy \
    openpyxl

Note: you may need to restart the kernel to use updated packages.


## 1. Import Libraries

In [1]:
import json
import random
import re
import sys
import time
from pathlib import Path
from typing import Any

import boto3
import numpy as np
import pandas as pd
import sagemaker
import sklearn

from sagemaker.jumpstart.model import JumpStartModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedKFold

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


## 2. Experiment Configuration

In [2]:
# =============================================================================
# Dataset settings
# =============================================================================

DATA_FILE = Path("./org_name_BC.csv")

TEXT_COLUMN = "Org_Name"
LABEL_COLUMN = "Class"
ID_COLUMN = "Instance_ID"

# =============================================================================
# Model and endpoint settings
# =============================================================================

MODEL_CHOICE = "llama3_3_70b_instruct"
MODEL_NAME = "Meta Llama 3.3 70B Instruct"

# This model ID uses AWS Trainium or Inferentia through SageMaker JumpStart.
MODEL_ID = "meta-textgeneration-llama-3-3-70b-instruct"

ENDPOINT_INSTANCE_TYPE = "ml.g5.48xlarge"

ENDPOINT_NAME = "llama3-3-70b-gov-nongov-bc"

ACCEPT_EULA = True

# =============================================================================
# Cross-validation settings
# =============================================================================

RANDOM_SEED = 42
N_SPLITS = 10

# Ten few-shot examples are drawn from the training portion of each fold.
N_FEW_SHOT_EXAMPLES = 10

# Use five examples from each class.
N_FEW_SHOT_PER_CLASS = 5

# =============================================================================
# Inference settings
# =============================================================================

MAX_NEW_TOKENS = 4

# Deterministic generation is preferred for classification.
TEMPERATURE = 0.0
DO_SAMPLE = False

MAX_RETRIES = 5
INITIAL_RETRY_DELAY = 3

# Add a short delay between requests if endpoint throttling occurs.
REQUEST_DELAY_SECONDS = 0.0

# =============================================================================
# Label settings
# =============================================================================

ID2LABEL = {
    0: "non-gov",
    1: "gov"
}

LABEL2ID = {
    "non-gov": 0,
    "gov": 1
}

# =============================================================================
# Output directories
# =============================================================================

OUTPUT_ROOT = Path(
    "./organization_classification_results_llama3"
)

ZERO_SHOT_RESULTS_DIR = (
    OUTPUT_ROOT / "zero_shot"
)

FEW_SHOT_RESULTS_DIR = (
    OUTPUT_ROOT / "few_shot"
)

CHECKPOINT_DIR = (
    OUTPUT_ROOT / "progress_checkpoints"
)

ZERO_SHOT_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEW_SHOT_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Dataset: {DATA_FILE}")
print(f"Model ID: {MODEL_ID}")
print(f"Endpoint instance: {ENDPOINT_INSTANCE_TYPE}")

Dataset: org_name_BC.csv
Model ID: meta-textgeneration-llama-3-3-70b-instruct
Endpoint instance: ml.g5.48xlarge


## 3. Set Random Seeds

In [3]:
def set_seed(seed: int) -> None:
    """Set Python and NumPy random seeds."""

    random.seed(seed)
    np.random.seed(seed)


set_seed(RANDOM_SEED)

## 4. Load and Validate the Dataset

In [4]:
# Load the CSV file while preserving multilingual characters.
df = pd.read_csv(
    DATA_FILE,
    encoding="utf-8-sig"
)

required_columns = {
    TEXT_COLUMN,
    LABEL_COLUMN
}

missing_columns = required_columns.difference(
    df.columns
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )

df = df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

df = df.dropna(
    subset=[TEXT_COLUMN, LABEL_COLUMN]
).reset_index(drop=True)

df[TEXT_COLUMN] = (
    df[TEXT_COLUMN]
    .astype(str)
    .str.strip()
)

df[LABEL_COLUMN] = pd.to_numeric(
    df[LABEL_COLUMN],
    errors="raise"
).astype(int)

if not set(
    df[LABEL_COLUMN].unique()
).issubset({0, 1}):
    raise ValueError(
        "Only class labels 0 and 1 are allowed."
    )

if df[TEXT_COLUMN].eq("").any():
    raise ValueError(
        "At least one organization name is empty."
    )

# Assign a stable identifier using the original row order.
df.insert(
    0,
    ID_COLUMN,
    np.arange(1, len(df) + 1)
)

print(f"Dataset shape: {df.shape}")

print("\nClass distribution:")
print(
    df[LABEL_COLUMN]
    .value_counts()
    .sort_index()
)

print(
    f"\nDuplicate organization names: "
    f"{df[TEXT_COLUMN].duplicated().sum()}"
)

display(df.head())

Dataset shape: (500, 3)

Class distribution:
Class
0    245
1    255
Name: count, dtype: int64

Duplicate organization names: 0


,Instance_ID,Org_Name,Class
0,1,Gulf of California Marine Program,0
1,2,Philips Lighting,0
2,3,Department of Natural Resource Ecology and Man...,0
3,4,"Ministry of Agriculture and Forestry, Finland",1
4,5,National Science Foundation,1


## 5. Create Identical Stratified Folds

In [5]:
stratified_kfold = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

texts = df[TEXT_COLUMN].to_numpy()
labels = df[LABEL_COLUMN].to_numpy()

fold_assignments = []

for fold_number, (
    train_indices,
    validation_indices
) in enumerate(
    stratified_kfold.split(texts, labels),
    start=1
):
    fold_assignments.append(
        {
            "fold": fold_number,
            "train_indices": train_indices,
            "validation_indices": validation_indices
        }
    )

print(
    f"Created {len(fold_assignments)} "
    f"stratified folds."
)

Created 10 stratified folds.


## 6. Deploy Llama 3.3 70B Instruct Endpoint

In [6]:
from sagemaker.jumpstart.model import JumpStartModel

# Create the SageMaker JumpStart model object.
jumpstart_model = JumpStartModel(
    model_id=MODEL_ID,
    model_version="*",
    instance_type=ENDPOINT_INSTANCE_TYPE,
    env={
        "OPTION_GPU_MEMORY_UTILIZATION": "0.90",
        "OPTION_MAX_MODEL_LEN": "8192",
        "OPTION_TENSOR_PARALLEL_DEGREE": "8",
        "OPTION_MAX_ROLLING_BATCH_SIZE": "4"
    }
)

# Deploy the model using the selected instance.
predictor = jumpstart_model.deploy(
    accept_eula=True,
    endpoint_name=ENDPOINT_NAME,
    instance_type=ENDPOINT_INSTANCE_TYPE
)

print(f"Endpoint deployed: {predictor.endpoint_name}")

Using model 'meta-textgeneration-llama-3-3-70b-instruct' with wildcard version identifier '*'. You can pin to version '1.52.0' for more stable results. Note that models may have different input/output signatures after a major version upgrade.
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: JumpStartModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
JumpStartModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1

----------------!

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: Predictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Predictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Endpoint deployed: llama3-3-70b-gov-nongov-bc


## 7. Define Classification Instructions

In [7]:
SYSTEM_INSTRUCTION = """
You are a binary classifier of funding organization names.

Classification rule:

Class 1:
The organization is a government organization, government agency,
government ministry, government department, national or subnational
public authority, public research council, or a funding organization
that is directly created, controlled, or operated by a government.

Class 0:
The organization is not government-related. This includes private
foundations, charities, companies, associations, universities,
nonprofit organizations, professional societies, and other
non-government organizations.

Decision rule:
Assign Class 1 only when the organization is clearly government-related.
If the organization does not satisfy the Class 1 definition, assign
Class 0.

Return exactly one character:
0 or 1

Do not provide explanations, punctuation, words, or additional text.
""".strip()

## 8. Define Prompt Construction Functions

In [8]:
def format_llama3_chat_prompt(
    system_instruction: str,
    user_content: str
) -> str:
    """Format a prompt using the Llama 3 chat template."""

    return (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_instruction}"
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_content}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

In [9]:
def build_zero_shot_prompt(
    organization_name: str
) -> str:
    """Build a zero-shot classification prompt."""

    user_content = (
        "Classify the following organization.\n\n"
        f"Organization name: {organization_name}\n\n"
        "Answer:"
    )

    return format_llama3_chat_prompt(
        system_instruction=SYSTEM_INSTRUCTION,
        user_content=user_content
    )

In [10]:
def build_few_shot_prompt(
    organization_name: str,
    few_shot_examples: pd.DataFrame
) -> str:
    """Build a few-shot classification prompt."""

    example_lines = []

    for example_number, (_, row) in enumerate(
        few_shot_examples.iterrows(),
        start=1
    ):
        example_lines.append(
            f"Example {example_number}\n"
            f"Organization name: {row[TEXT_COLUMN]}\n"
            f"Class: {int(row[LABEL_COLUMN])}"
        )

    examples_text = "\n\n".join(
        example_lines
    )

    user_content = (
        "Use the labeled examples below as guidance.\n\n"
        f"{examples_text}\n\n"
        "Now classify the following organization.\n\n"
        f"Organization name: {organization_name}\n\n"
        "Answer:"
    )

    return format_llama3_chat_prompt(
        system_instruction=SYSTEM_INSTRUCTION,
        user_content=user_content
    )

## 9. Select Few-Shot Examples from Each Training Fold

In [11]:
def sample_balanced_few_shot_examples(
    training_df: pd.DataFrame,
    fold_number: int
) -> pd.DataFrame:
    """
    Sample five examples from each class using only the training fold.
    """

    fold_seed = RANDOM_SEED + fold_number

    class_0_examples = (
        training_df[
            training_df[LABEL_COLUMN] == 0
        ]
        .sample(
            n=N_FEW_SHOT_PER_CLASS,
            random_state=fold_seed
        )
    )

    class_1_examples = (
        training_df[
            training_df[LABEL_COLUMN] == 1
        ]
        .sample(
            n=N_FEW_SHOT_PER_CLASS,
            random_state=fold_seed
        )
    )

    few_shot_examples = pd.concat(
        [
            class_0_examples,
            class_1_examples
        ],
        ignore_index=True
    )

    # Shuffle the example order to avoid always presenting one class first.
    few_shot_examples = (
        few_shot_examples
        .sample(
            frac=1,
            random_state=fold_seed
        )
        .reset_index(drop=True)
    )

    if len(few_shot_examples) != N_FEW_SHOT_EXAMPLES:
        raise RuntimeError(
            "The number of few-shot examples is incorrect."
        )

    return few_shot_examples

## 10. Define Endpoint Inference and Output Parsing

In [12]:
def extract_generated_text(
    response: Any
) -> str:
    """Extract generated text from common JumpStart response formats."""

    if isinstance(response, str):
        return response

    if isinstance(response, bytes):
        return response.decode(
            "utf-8",
            errors="replace"
        )

    if isinstance(response, dict):
        for key in [
            "generated_text",
            "generation",
            "text",
            "output"
        ]:
            if key in response:
                return str(response[key])

    if isinstance(response, list) and response:
        first_item = response[0]

        if isinstance(first_item, str):
            return first_item

        if isinstance(first_item, dict):
            for key in [
                "generated_text",
                "generation",
                "text",
                "output"
            ]:
                if key in first_item:
                    return str(first_item[key])

    return str(response)

In [13]:
def parse_binary_class(
    generated_text: str
) -> tuple[int, bool]:
    """
    Parse the first standalone binary label.

    Returns:
        predicted_class: Parsed class or 0 when no valid class is found.
        valid_format: Whether an explicit 0 or 1 was successfully parsed.
    """

    cleaned_text = generated_text.strip()

    # Prefer a response containing only the requested label.
    if cleaned_text in {"0", "1"}:
        return int(cleaned_text), True

    # Remove common Llama end-of-turn markers.
    cleaned_text = cleaned_text.replace(
        "<|eot_id|>",
        " "
    ).strip()

    if cleaned_text in {"0", "1"}:
        return int(cleaned_text), True

    # Recover the first standalone binary label from a noncompliant response.
    match = re.search(
        r"(?<!\d)([01])(?!\d)",
        cleaned_text
    )

    if match:
        return int(match.group(1)), True

    # Apply the conservative decision rule when the output cannot be parsed.
    return 0, False

In [14]:
def invoke_classifier(
    prompt: str
) -> dict:
    """Invoke the SageMaker endpoint with retry handling."""

    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": MAX_NEW_TOKENS,
            "temperature": TEMPERATURE,
            "do_sample": DO_SAMPLE,
            "return_full_text": False,
            "stop": [
                "<|eot_id|>",
                "\n"
            ]
        }
    }

    last_exception = None

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):
        try:
            response = predictor.predict(
                payload
            )

            generated_text = extract_generated_text(
                response
            )

            predicted_class, valid_format = (
                parse_binary_class(
                    generated_text
                )
            )

            if REQUEST_DELAY_SECONDS > 0:
                time.sleep(
                    REQUEST_DELAY_SECONDS
                )

            return {
                "predicted_class": predicted_class,
                "raw_response": generated_text,
                "valid_format": valid_format,
                "attempts": attempt
            }

        except Exception as exception:
            last_exception = exception

            if attempt == MAX_RETRIES:
                break

            delay = (
                INITIAL_RETRY_DELAY
                * (2 ** (attempt - 1))
            )

            print(
                f"Endpoint call failed on attempt "
                f"{attempt}. Retrying in {delay} seconds."
            )

            time.sleep(delay)

    raise RuntimeError(
        "Endpoint invocation failed after all retries."
    ) from last_exception

## 11. Test the Endpoint

In [15]:
test_prompt = build_zero_shot_prompt(
    "United States Department of Energy"
)

test_result = invoke_classifier(
    test_prompt
)

print(test_result)

{'predicted_class': 1, 'raw_response': '1', 'valid_format': True, 'attempts': 1}


## 12. Define Evaluation Functions

In [16]:
def calculate_fold_metrics(
    true_labels: np.ndarray,
    predicted_labels: np.ndarray,
    fold_number: int
) -> dict:
    """Calculate class-level and aggregate metrics for one fold."""

    class_precision, class_recall, class_f1, class_support = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1],
            average=None,
            zero_division=0
        )
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1],
            average="macro",
            zero_division=0
        )
    )

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1],
            average="weighted",
            zero_division=0
        )
    )

    return {
        "fold": fold_number,
        "accuracy": float(
            accuracy_score(
                true_labels,
                predicted_labels
            )
        ),

        "class_0_precision": float(
            class_precision[0]
        ),
        "class_0_recall": float(
            class_recall[0]
        ),
        "class_0_f1": float(
            class_f1[0]
        ),
        "class_0_support": int(
            class_support[0]
        ),

        "class_1_precision": float(
            class_precision[1]
        ),
        "class_1_recall": float(
            class_recall[1]
        ),
        "class_1_f1": float(
            class_f1[1]
        ),
        "class_1_support": int(
            class_support[1]
        ),

        "macro_precision": float(
            macro_precision
        ),
        "macro_recall": float(
            macro_recall
        ),
        "macro_f1": float(
            macro_f1
        ),

        "weighted_precision": float(
            weighted_precision
        ),
        "weighted_recall": float(
            weighted_recall
        ),
        "weighted_f1": float(
            weighted_f1
        )
    }

## 13. Run Zero-Shot 10-Fold Evaluation

In [17]:
zero_shot_fold_metrics = []
zero_shot_confusion_matrices = []
zero_shot_predictions = []

for fold_info in fold_assignments:
    fold_number = fold_info["fold"]
    validation_indices = fold_info[
        "validation_indices"
    ]

    validation_df = df.iloc[
        validation_indices
    ].copy()

    print("=" * 100)
    print(
        f"Zero-shot fold {fold_number} "
        f"of {N_SPLITS}"
    )
    print("=" * 100)

    fold_records = []

    for position, (_, row) in enumerate(
        validation_df.iterrows(),
        start=1
    ):
        organization_name = row[
            TEXT_COLUMN
        ]

        prompt = build_zero_shot_prompt(
            organization_name
        )

        result = invoke_classifier(
            prompt
        )

        true_class = int(
            row[LABEL_COLUMN]
        )

        predicted_class = int(
            result["predicted_class"]
        )

        record = {
            ID_COLUMN: int(row[ID_COLUMN]),
            TEXT_COLUMN: organization_name,
            "True_Class": true_class,
            "True_Label": ID2LABEL[true_class],
            "Predicted_Class": predicted_class,
            "Predicted_Label": ID2LABEL[
                predicted_class
            ],
            "Correct_Prediction": bool(
                true_class == predicted_class
            ),
            "Valid_Output_Format": bool(
                result["valid_format"]
            ),
            "Raw_Response": result[
                "raw_response"
            ],
            "Invocation_Attempts": int(
                result["attempts"]
            ),
            "Fold": fold_number,
            "Prompting_Method": "zero_shot",
            "Model_Choice": MODEL_CHOICE,
            "Model_Name": MODEL_NAME
        }

        fold_records.append(record)
        zero_shot_predictions.append(record)

        if position % 10 == 0:
            print(
                f"Completed {position}/"
                f"{len(validation_df)} instances."
            )

    # Save progress immediately after each fold.
    fold_checkpoint_df = pd.DataFrame(
        fold_records
    )

    fold_checkpoint_file = (
        CHECKPOINT_DIR
        / f"zero_shot_fold_{fold_number}.csv"
    )

    fold_checkpoint_df.to_csv(
        fold_checkpoint_file,
        index=False,
        encoding="utf-8-sig"
    )

    true_labels_fold = fold_checkpoint_df[
        "True_Class"
    ].to_numpy()

    predicted_labels_fold = fold_checkpoint_df[
        "Predicted_Class"
    ].to_numpy()

    fold_metrics = calculate_fold_metrics(
        true_labels=true_labels_fold,
        predicted_labels=predicted_labels_fold,
        fold_number=fold_number
    )

    fold_metrics[
        "invalid_output_count"
    ] = int(
        (~fold_checkpoint_df[
            "Valid_Output_Format"
        ]).sum()
    )

    zero_shot_fold_metrics.append(
        fold_metrics
    )

    fold_cm = confusion_matrix(
        true_labels_fold,
        predicted_labels_fold,
        labels=[0, 1]
    )

    zero_shot_confusion_matrices.append(
        {
            "fold": fold_number,
            "true_0_pred_0": int(
                fold_cm[0, 0]
            ),
            "true_0_pred_1": int(
                fold_cm[0, 1]
            ),
            "true_1_pred_0": int(
                fold_cm[1, 0]
            ),
            "true_1_pred_1": int(
                fold_cm[1, 1]
            )
        }
    )

    print(
        f"Fold {fold_number} accuracy: "
        f"{fold_metrics['accuracy']:.4f}"
    )

Zero-shot fold 1 of 10
Completed 10/50 instances.
Completed 20/50 instances.
Completed 30/50 instances.
Completed 40/50 instances.
Completed 50/50 instances.
Fold 1 accuracy: 0.9600
Zero-shot fold 2 of 10
Completed 10/50 instances.
Completed 20/50 instances.
Completed 30/50 instances.
Completed 40/50 instances.
Completed 50/50 instances.
Fold 2 accuracy: 0.9000
Zero-shot fold 3 of 10
Completed 10/50 instances.
Completed 20/50 instances.
Completed 30/50 instances.
Completed 40/50 instances.
Completed 50/50 instances.
Fold 3 accuracy: 0.9400
Zero-shot fold 4 of 10
Completed 10/50 instances.
Completed 20/50 instances.
Completed 30/50 instances.
Completed 40/50 instances.
Completed 50/50 instances.
Fold 4 accuracy: 0.8600
Zero-shot fold 5 of 10
Completed 10/50 instances.
Completed 20/50 instances.
Completed 30/50 instances.
Completed 40/50 instances.
Completed 50/50 instances.
Fold 5 accuracy: 0.9600
Zero-shot fold 6 of 10
Completed 10/50 instances.
Completed 20/50 instances.
Completed 30/

## 14. Run Few-Shot 10-Fold Evaluation

In [18]:
few_shot_fold_metrics = []
few_shot_confusion_matrices = []
few_shot_predictions = []
few_shot_example_records = []

for fold_info in fold_assignments:
    fold_number = fold_info["fold"]

    train_indices = fold_info[
        "train_indices"
    ]

    validation_indices = fold_info[
        "validation_indices"
    ]

    training_df = df.iloc[
        train_indices
    ].copy()

    validation_df = df.iloc[
        validation_indices
    ].copy()

    few_shot_examples = (
        sample_balanced_few_shot_examples(
            training_df=training_df,
            fold_number=fold_number
        )
    )

    # Save the selected examples for reproducibility.
    examples_for_saving = (
        few_shot_examples[
            [
                ID_COLUMN,
                TEXT_COLUMN,
                LABEL_COLUMN
            ]
        ]
        .copy()
    )

    examples_for_saving.insert(
        0,
        "Fold",
        fold_number
    )

    examples_for_saving.insert(
        1,
        "Example_Order",
        np.arange(
            1,
            len(examples_for_saving) + 1
        )
    )

    few_shot_example_records.extend(
        examples_for_saving.to_dict(
            orient="records"
        )
    )

    print("=" * 100)
    print(
        f"Few-shot fold {fold_number} "
        f"of {N_SPLITS}"
    )
    print("=" * 100)

    print("Selected few-shot examples:")
    display(
        examples_for_saving
    )

    fold_records = []

    for position, (_, row) in enumerate(
        validation_df.iterrows(),
        start=1
    ):
        organization_name = row[
            TEXT_COLUMN
        ]

        prompt = build_few_shot_prompt(
            organization_name=organization_name,
            few_shot_examples=few_shot_examples
        )

        result = invoke_classifier(
            prompt
        )

        true_class = int(
            row[LABEL_COLUMN]
        )

        predicted_class = int(
            result["predicted_class"]
        )

        record = {
            ID_COLUMN: int(row[ID_COLUMN]),
            TEXT_COLUMN: organization_name,
            "True_Class": true_class,
            "True_Label": ID2LABEL[true_class],
            "Predicted_Class": predicted_class,
            "Predicted_Label": ID2LABEL[
                predicted_class
            ],
            "Correct_Prediction": bool(
                true_class == predicted_class
            ),
            "Valid_Output_Format": bool(
                result["valid_format"]
            ),
            "Raw_Response": result[
                "raw_response"
            ],
            "Invocation_Attempts": int(
                result["attempts"]
            ),
            "Fold": fold_number,
            "Prompting_Method": "few_shot",
            "Few_Shot_Example_Count": (
                N_FEW_SHOT_EXAMPLES
            ),
            "Model_Choice": MODEL_CHOICE,
            "Model_Name": MODEL_NAME
        }

        fold_records.append(record)
        few_shot_predictions.append(record)

        if position % 10 == 0:
            print(
                f"Completed {position}/"
                f"{len(validation_df)} instances."
            )

    fold_checkpoint_df = pd.DataFrame(
        fold_records
    )

    fold_checkpoint_file = (
        CHECKPOINT_DIR
        / f"few_shot_fold_{fold_number}.csv"
    )

    fold_checkpoint_df.to_csv(
        fold_checkpoint_file,
        index=False,
        encoding="utf-8-sig"
    )

    true_labels_fold = fold_checkpoint_df[
        "True_Class"
    ].to_numpy()

    predicted_labels_fold = fold_checkpoint_df[
        "Predicted_Class"
    ].to_numpy()

    fold_metrics = calculate_fold_metrics(
        true_labels=true_labels_fold,
        predicted_labels=predicted_labels_fold,
        fold_number=fold_number
    )

    fold_metrics[
        "invalid_output_count"
    ] = int(
        (~fold_checkpoint_df[
            "Valid_Output_Format"
        ]).sum()
    )

    few_shot_fold_metrics.append(
        fold_metrics
    )

    fold_cm = confusion_matrix(
        true_labels_fold,
        predicted_labels_fold,
        labels=[0, 1]
    )

    few_shot_confusion_matrices.append(
        {
            "fold": fold_number,
            "true_0_pred_0": int(
                fold_cm[0, 0]
            ),
            "true_0_pred_1": int(
                fold_cm[0, 1]
            ),
            "true_1_pred_0": int(
                fold_cm[1, 0]
            ),
            "true_1_pred_1": int(
                fold_cm[1, 1]
            )
        }
    )

    print(
        f"Fold {fold_number} accuracy: "
        f"{fold_metrics['accuracy']:.4f}"
    )

Few-shot fold 1 of 10
Selected few-shot examples:


,Fold,Example_Order,Instance_ID,Org_Name,Class
0,1,1,308,Conservation International's southern Africa's...,0
1,1,2,373,Educational and scientific research program fo...,1
2,1,3,143,Swiss National Science Foundation,1
3,1,4,251,United States Fish and Wildlife Service,1
4,1,5,12,Copernicus Foundation,0
5,1,6,287,Conservation and Management of the Eastern Arc...,1
6,1,7,51,David and Lucile Packard FoundationThe David &...,0
7,1,8,282,Agence Nationale de la Recherche through GEMS ...,1
8,1,9,29,"College of Liberal Arts and Social Sciences, C...",0
9,1,10,180,Tall Timbers Research Station,0


Completed 10/50 instances.
Completed 20/50 instances.
Completed 30/50 instances.
Completed 40/50 instances.
Completed 50/50 instances.
Fold 1 accuracy: 0.9600
Few-shot fold 2 of 10
Selected few-shot examples:


,Fold,Example_Order,Instance_ID,Org_Name,Class
0,2,1,371,Instituto Bioclon,0
1,2,2,130,Brazilian Center for Improvement of Higher Edu...,1
2,2,3,163,Worland BLM Field Office,1
3,2,4,303,Caterpillar company,0
4,2,5,399,Adaptive Management project under Local and Gl...,1
5,2,6,196,Florida Department of Environmental Protection...,1
6,2,7,150,Uinta County Predator Management Board,1
7,2,8,261,FACCE-JPI DEVIL project,0
8,2,9,231,Texas A&M RAMSES research group,0
9,2,10,368,Lake Forest Garden Club,0


Completed 10/50 instances.
Completed 20/50 instances.
Completed 30/50 instances.
Completed 40/50 instances.
Completed 50/50 instances.
Fold 2 accuracy: 0.8600
Few-shot fold 3 of 10
Selected few-shot examples:


,Fold,Example_Order,Instance_ID,Org_Name,Class
0,3,1,379,Andean Bear Foundation,0
1,3,2,206,Modeling and Simulation Graduate Research Fell...,0
2,3,3,384,Technology Center of Japan,0
3,3,4,373,Educational and scientific research program fo...,1
4,3,5,444,Bureau of Land Management,1
5,3,6,460,Province R&D Program of Jiangsu,1
6,3,7,369,Guangdong Foundation for Program of Science an...,1
7,3,8,462,CATER 2006-3302 in Korea,1
8,3,9,448,Atlantic Salmon Trust,0
9,3,10,201,Banaras Hindu University,0


... (11 outputs left out)

,Fold,Example_Order,Instance_ID,Org_Name,Class
0,9,1,192,INCAS3,0
1,9,2,371,Instituto Bioclon,0
2,9,3,496,Starting Foundation for PhD of Guangxi University,0
3,9,4,387,Biotechnology and Biological Sciences Research...,1
4,9,5,128,Comision Nacional de Investigacion Cientifica ...,1
5,9,6,155,OpenNESS Project,1
6,9,7,38,Society of Canadian Ornithologists,0
7,9,8,164,School of Forest Resources and Environmental S...,0
8,9,9,19,Northern Ireland Environment Agency,1
9,9,10,84,Royal Botanic Garden Edinburgh,1


Completed 10/50 instances.
Completed 20/50 instances.
Completed 30/50 instances.
Completed 40/50 instances.
Completed 50/50 instances.
Fold 9 accuracy: 0.9400
Few-shot fold 10 of 10
Selected few-shot examples:


,Fold,Example_Order,Instance_ID,Org_Name,Class
0,10,1,484,Revealing Hominid Origins Initiative,1
1,10,2,198,Imperial Oil Ltd.,0
2,10,3,106,'Mov-It' ANR grant,1
3,10,4,274,research of Dagmar Haase,0
4,10,5,237,Landesjagdabgabe Baden-Wurttemberg,0
5,10,6,448,Atlantic Salmon Trust,0
6,10,7,261,FACCE-JPI DEVIL project,0
7,10,8,246,Murray Darling Basin Authority,1
8,10,9,177,Planning Project of Science and Technology of ...,1
9,10,10,316,Comision Nacional para la Biodiversidad,1


Completed 10/50 instances.
Completed 20/50 instances.
Completed 30/50 instances.
Completed 40/50 instances.
Completed 50/50 instances.
Fold 10 accuracy: 0.8600


## 15. Save Few-Shot Examples

In [19]:
few_shot_examples_df = pd.DataFrame(
    few_shot_example_records
)

few_shot_examples_file = (
    FEW_SHOT_RESULTS_DIR
    / "llama3_few_shot_examples_by_fold.csv"
)

few_shot_examples_df.to_csv(
    few_shot_examples_file,
    index=False,
    encoding="utf-8-sig"
)

display(
    few_shot_examples_df.head(20)
)

print(
    f"Few-shot examples saved to: "
    f"{few_shot_examples_file}"
)

,Fold,Example_Order,Instance_ID,Org_Name,Class
0,1,1,308,Conservation International's southern Africa's...,0
1,1,2,373,Educational and scientific research program fo...,1
2,1,3,143,Swiss National Science Foundation,1
3,1,4,251,United States Fish and Wildlife Service,1
4,1,5,12,Copernicus Foundation,0
5,1,6,287,Conservation and Management of the Eastern Arc...,1
6,1,7,51,David and Lucile Packard FoundationThe David &...,0
7,1,8,282,Agence Nationale de la Recherche through GEMS ...,1
8,1,9,29,"College of Liberal Arts and Social Sciences, C...",0
9,1,10,180,Tall Timbers Research Station,0


Few-shot examples saved to: organization_classification_results_llama3/few_shot/llama3_few_shot_examples_by_fold.csv


## 16. Define Result Export Function

In [20]:
METRIC_COLUMNS = [
    "accuracy",

    "class_0_precision",
    "class_0_recall",
    "class_0_f1",

    "class_1_precision",
    "class_1_recall",
    "class_1_f1",

    "macro_precision",
    "macro_recall",
    "macro_f1",

    "weighted_precision",
    "weighted_recall",
    "weighted_f1"
]


def export_experiment_results(
    method_name: str,
    fold_metrics_list: list[dict],
    confusion_matrix_list: list[dict],
    prediction_records: list[dict],
    results_directory: Path
) -> dict:
    """Save all evaluation outputs for one prompting method."""

    # -------------------------------------------------------------------------
    # Fold-level metrics
    # -------------------------------------------------------------------------

    fold_metrics_df = (
        pd.DataFrame(fold_metrics_list)
        .sort_values("fold")
    )

    fold_metrics_file = (
        results_directory
        / f"{method_name}_10fold_metrics.csv"
    )

    fold_metrics_df.to_csv(
        fold_metrics_file,
        index=False,
        encoding="utf-8-sig"
    )

    # -------------------------------------------------------------------------
    # Aggregate metrics
    # -------------------------------------------------------------------------

    aggregate_metrics_df = pd.DataFrame(
        {
            "Metric": METRIC_COLUMNS,

            "Mean": [
                fold_metrics_df[
                    metric
                ].mean()
                for metric in METRIC_COLUMNS
            ],

            "Standard_Deviation": [
                fold_metrics_df[
                    metric
                ].std(ddof=1)
                for metric in METRIC_COLUMNS
            ],

            "Minimum": [
                fold_metrics_df[
                    metric
                ].min()
                for metric in METRIC_COLUMNS
            ],

            "Maximum": [
                fold_metrics_df[
                    metric
                ].max()
                for metric in METRIC_COLUMNS
            ]
        }
    )

    numeric_columns = [
        "Mean",
        "Standard_Deviation",
        "Minimum",
        "Maximum"
    ]

    aggregate_metrics_df[
        numeric_columns
    ] = aggregate_metrics_df[
        numeric_columns
    ].round(4)

    aggregate_metrics_file = (
        results_directory
        / f"{method_name}_10fold_aggregate_metrics.csv"
    )

    aggregate_metrics_df.to_csv(
        aggregate_metrics_file,
        index=False,
        encoding="utf-8-sig"
    )

    # -------------------------------------------------------------------------
    # Fold-level confusion matrices
    # -------------------------------------------------------------------------

    confusion_matrices_df = (
        pd.DataFrame(
            confusion_matrix_list
        )
        .sort_values("fold")
    )

    confusion_matrices_file = (
        results_directory
        / f"{method_name}_10fold_confusion_matrices.csv"
    )

    confusion_matrices_df.to_csv(
        confusion_matrices_file,
        index=False,
        encoding="utf-8-sig"
    )

    # -------------------------------------------------------------------------
    # Instance-level out-of-fold predictions
    # -------------------------------------------------------------------------

    predictions_df = (
        pd.DataFrame(
            prediction_records
        )
        .sort_values(ID_COLUMN)
    )

    if len(predictions_df) != len(df):
        raise RuntimeError(
            f"{method_name}: The number of predictions "
            f"does not match the dataset size."
        )

    if predictions_df[
        ID_COLUMN
    ].duplicated().any():
        raise RuntimeError(
            f"{method_name}: Duplicate OOF predictions "
            f"were detected."
        )

    predictions_file = (
        results_directory
        / f"{method_name}_10fold_instance_predictions.csv"
    )

    predictions_df.to_csv(
        predictions_file,
        index=False,
        encoding="utf-8-sig"
    )

    # -------------------------------------------------------------------------
    # Pooled out-of-fold evaluation
    # -------------------------------------------------------------------------

    true_labels = predictions_df[
        "True_Class"
    ].to_numpy()

    predicted_labels = predictions_df[
        "Predicted_Class"
    ].to_numpy()

    report_dictionary = classification_report(
        true_labels,
        predicted_labels,
        labels=[0, 1],
        target_names=[
            "non-gov",
            "gov"
        ],
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        true_labels,
        predicted_labels,
        labels=[0, 1],
        target_names=[
            "non-gov",
            "gov"
        ],
        digits=4,
        zero_division=0
    )

    report_df = (
        pd.DataFrame(
            report_dictionary
        )
        .transpose()
        .reset_index()
        .rename(
            columns={
                "index": "Metric"
            }
        )
    )

    report_csv_file = (
        results_directory
        / f"{method_name}_overall_oof_report.csv"
    )

    report_text_file = (
        results_directory
        / f"{method_name}_overall_oof_report.txt"
    )

    report_df.to_csv(
        report_csv_file,
        index=False,
        encoding="utf-8-sig"
    )

    with open(
        report_text_file,
        "w",
        encoding="utf-8"
    ) as file:
        file.write(report_text)

    overall_cm = confusion_matrix(
        true_labels,
        predicted_labels,
        labels=[0, 1]
    )

    overall_cm_df = pd.DataFrame(
        overall_cm,
        index=[
            "True_0_non_gov",
            "True_1_gov"
        ],
        columns=[
            "Predicted_0_non_gov",
            "Predicted_1_gov"
        ]
    )

    overall_cm_file = (
        results_directory
        / f"{method_name}_overall_oof_confusion_matrix.csv"
    )

    overall_cm_df.to_csv(
        overall_cm_file,
        encoding="utf-8-sig"
    )

    # -------------------------------------------------------------------------
    # Misclassified instances
    # -------------------------------------------------------------------------

    misclassified_df = predictions_df[
        ~predictions_df[
            "Correct_Prediction"
        ]
    ].copy()

    misclassified_file = (
        results_directory
        / f"{method_name}_misclassified_instances.csv"
    )

    misclassified_df.to_csv(
        misclassified_file,
        index=False,
        encoding="utf-8-sig"
    )

    # -------------------------------------------------------------------------
    # Invalidly formatted model responses
    # -------------------------------------------------------------------------

    invalid_outputs_df = predictions_df[
        ~predictions_df[
            "Valid_Output_Format"
        ]
    ].copy()

    invalid_outputs_file = (
        results_directory
        / f"{method_name}_invalid_outputs.csv"
    )

    invalid_outputs_df.to_csv(
        invalid_outputs_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"\n{method_name} overall OOF performance:"
    )

    print(report_text)

    print("Overall confusion matrix:")
    print(overall_cm)

    return {
        "fold_metrics": fold_metrics_df,
        "aggregate_metrics": aggregate_metrics_df,
        "predictions": predictions_df,
        "classification_report": report_df,
        "confusion_matrix": overall_cm_df,
        "misclassified": misclassified_df,
        "invalid_outputs": invalid_outputs_df
    }

## 17. Export Zero-Shot Results

In [21]:
zero_shot_results = export_experiment_results(
    method_name="zero_shot",
    fold_metrics_list=zero_shot_fold_metrics,
    confusion_matrix_list=zero_shot_confusion_matrices,
    prediction_records=zero_shot_predictions,
    results_directory=ZERO_SHOT_RESULTS_DIR
)

display(
    zero_shot_results[
        "aggregate_metrics"
    ]
)


zero_shot overall OOF performance:
              precision    recall  f1-score   support

     non-gov     0.8794    0.9224    0.9004       245
         gov     0.9218    0.8784    0.8996       255

    accuracy                         0.9000       500
   macro avg     0.9006    0.9004    0.9000       500
weighted avg     0.9010    0.9000    0.9000       500

Overall confusion matrix:
[[226  19]
 [ 31 224]]


,Metric,Mean,Standard_Deviation,Minimum,Maximum
0,accuracy,0.9000,0.0566,0.8000,0.9600
1,class_0_precision,0.8819,0.0738,0.7692,1.0000
2,class_0_recall,0.9223,0.0569,0.8333,1.0000
3,class_0_f1,0.9006,0.0565,0.8000,0.9600
4,class_1_precision,0.9228,0.0554,0.8333,1.0000
5,class_1_recall,0.8792,0.0760,0.7692,1.0000
6,class_1_f1,0.8993,0.0569,0.8000,0.9615
7,macro_precision,0.9024,0.0567,0.8013,0.9630
8,macro_recall,0.9008,0.0564,0.8013,0.9600
9,macro_f1,0.8999,0.0566,0.8000,0.9600


## 18. Export Few-Shot Results

In [22]:
few_shot_results = export_experiment_results(
    method_name="few_shot",
    fold_metrics_list=few_shot_fold_metrics,
    confusion_matrix_list=few_shot_confusion_matrices,
    prediction_records=few_shot_predictions,
    results_directory=FEW_SHOT_RESULTS_DIR
)

display(
    few_shot_results[
        "aggregate_metrics"
    ]
)


few_shot overall OOF performance:
              precision    recall  f1-score   support

     non-gov     0.8657    0.9469    0.9045       245
         gov     0.9440    0.8588    0.8994       255

    accuracy                         0.9020       500
   macro avg     0.9048    0.9029    0.9019       500
weighted avg     0.9056    0.9020    0.9019       500

Overall confusion matrix:
[[232  13]
 [ 36 219]]


,Metric,Mean,Standard_Deviation,Minimum,Maximum
0,accuracy,0.9020,0.0545,0.8200,0.9800
1,class_0_precision,0.8704,0.0748,0.7419,1.0000
2,class_0_recall,0.9472,0.0471,0.8750,1.0000
3,class_0_f1,0.9056,0.0507,0.8364,0.9796
4,class_1_precision,0.9446,0.0477,0.8750,1.0000
5,class_1_recall,0.8591,0.0883,0.6923,1.0000
6,class_1_f1,0.8977,0.0596,0.8000,0.9804
7,macro_precision,0.9075,0.0509,0.8447,0.9800
8,macro_recall,0.9031,0.0539,0.8253,0.9808
9,macro_f1,0.9017,0.0549,0.8182,0.9800


## 19. Compare Zero-Shot and Few-Shot Results

In [23]:
zero_summary = zero_shot_results[
    "aggregate_metrics"
][
    [
        "Metric",
        "Mean",
        "Standard_Deviation"
    ]
].copy()

zero_summary = zero_summary.rename(
    columns={
        "Mean": "Zero_Shot_Mean",
        "Standard_Deviation": "Zero_Shot_SD"
    }
)

few_summary = few_shot_results[
    "aggregate_metrics"
][
    [
        "Metric",
        "Mean",
        "Standard_Deviation"
    ]
].copy()

few_summary = few_summary.rename(
    columns={
        "Mean": "Few_Shot_Mean",
        "Standard_Deviation": "Few_Shot_SD"
    }
)

comparison_df = zero_summary.merge(
    few_summary,
    on="Metric",
    how="inner"
)

comparison_df[
    "Few_Minus_Zero"
] = (
    comparison_df[
        "Few_Shot_Mean"
    ]
    - comparison_df[
        "Zero_Shot_Mean"
    ]
).round(4)

comparison_file = (
    OUTPUT_ROOT
    / "zero_shot_vs_few_shot_comparison.csv"
)

comparison_df.to_csv(
    comparison_file,
    index=False,
    encoding="utf-8-sig"
)

display(comparison_df)

print(
    f"Comparison saved to: "
    f"{comparison_file}"
)

,Metric,Zero_Shot_Mean,Zero_Shot_SD,Few_Shot_Mean,Few_Shot_SD,Few_Minus_Zero
0,accuracy,0.9000,0.0566,0.9020,0.0545,0.0020
1,class_0_precision,0.8819,0.0738,0.8704,0.0748,-0.0115
2,class_0_recall,0.9223,0.0569,0.9472,0.0471,0.0249
3,class_0_f1,0.9006,0.0565,0.9056,0.0507,0.0050
4,class_1_precision,0.9228,0.0554,0.9446,0.0477,0.0218
5,class_1_recall,0.8792,0.0760,0.8591,0.0883,-0.0201
6,class_1_f1,0.8993,0.0569,0.8977,0.0596,-0.0016
7,macro_precision,0.9024,0.0567,0.9075,0.0509,0.0051
8,macro_recall,0.9008,0.0564,0.9031,0.0539,0.0023
9,macro_f1,0.8999,0.0566,0.9017,0.0549,0.0018


Comparison saved to: organization_classification_results_llama3/zero_shot_vs_few_shot_comparison.csv


## 20. Save Experiment Metadata

In [24]:
experiment_metadata = {
    "model_choice": MODEL_CHOICE,
    "model_name": MODEL_NAME,
    "model_id": MODEL_ID,
    "endpoint_instance_type": ENDPOINT_INSTANCE_TYPE,

    "dataset_file": str(DATA_FILE),
    "text_column": TEXT_COLUMN,
    "label_column": LABEL_COLUMN,

    "number_of_instances": int(
        len(df)
    ),

    "class_0_count": int(
        (df[LABEL_COLUMN] == 0).sum()
    ),

    "class_1_count": int(
        (df[LABEL_COLUMN] == 1).sum()
    ),

    "cross_validation_folds": N_SPLITS,

    "few_shot_examples_per_fold": (
        N_FEW_SHOT_EXAMPLES
    ),

    "few_shot_examples_per_class": (
        N_FEW_SHOT_PER_CLASS
    ),

    "random_seed": RANDOM_SEED,

    "temperature": TEMPERATURE,
    "do_sample": DO_SAMPLE,
    "max_new_tokens": MAX_NEW_TOKENS,

    "decision_rule": (
        "Assign class 1 only when clearly government-related; "
        "otherwise assign class 0."
    ),

    "sagemaker_version": (
        sagemaker.__version__
    ),

    "scikit_learn_version": (
        sklearn.__version__
    ),

    "pandas_version": (
        pd.__version__
    ),

    "numpy_version": (
        np.__version__
    ),

    "python_version": sys.version
}

metadata_file = (
    OUTPUT_ROOT
    / "experiment_metadata.json"
)

with open(
    metadata_file,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        experiment_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )

print(
    f"Metadata saved to: "
    f"{metadata_file}"
)

Metadata saved to: organization_classification_results_llama3/experiment_metadata.json


## 21. Delete the SageMaker Endpoint

In [25]:
# Delete the real-time endpoint to stop endpoint instance charges.
predictor.delete_endpoint(
    delete_endpoint_config=True
)

print(
    f"Endpoint deleted: {ENDPOINT_NAME}"
)

# Delete the associated SageMaker model object when supported by the SDK.
try:
    predictor.delete_model()

    print(
        "Associated SageMaker model object deleted."
    )

except Exception as exception:
    print(
        "The endpoint was deleted, but automatic model-object "
        "deletion was not completed."
    )

    print(
        f"Cleanup message: {exception}"
    )

Endpoint deleted: llama3-3-70b-gov-nongov-bc
The endpoint was deleted, but automatic model-object deletion was not completed.
Cleanup message: An error occurred (ValidationException) when calling the DescribeEndpointConfig operation: Could not find endpoint configuration "llama3-3-70b-gov-nongov-bc".


In [26]:
sagemaker_client = boto3.client(
    "sagemaker"
)

remaining_endpoints = (
    sagemaker_client
    .list_endpoints(
        NameContains=ENDPOINT_NAME
    )
    .get(
        "Endpoints",
        []
    )
)

print(remaining_endpoints)

[{'EndpointName': 'llama3-3-70b-gov-nongov-bc', 'EndpointArn': 'arn:aws:sagemaker:us-east-2:083883994478:endpoint/llama3-3-70b-gov-nongov-bc', 'CreationTime': datetime.datetime(2026, 8, 8, 21, 28, 56, 909000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2026, 8, 8, 21, 41, 32, 763000, tzinfo=tzlocal()), 'EndpointStatus': 'Deleting'}]
